In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score, precision_score, recall_score
from mlsmote_updated import load_genres, imbalance, mlsmote, predict_per_genre

SEED = 1001
K = 5
THRESHOLD = 0.3
DATA_PATH = next((p for p in [Path('cleaned_animes.csv'), Path('../cleaned_animes.csv'), Path('/mnt/data/cleaned_animes.csv')] if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Place cleaned_animes.csv in this directory or its parent.')
print('Using:', DATA_PATH)

Using: ../cleaned_animes.csv


In [2]:
df = pd.read_csv(DATA_PATH)
if 'text' not in df.columns:
    if {'title_english', 'synopsis'}.issubset(df.columns):
        df['text'] = df['title_english'].fillna('') + ' ' + df['synopsis'].fillna('')
    else:
        raise ValueError('Missing text column')
df['parsed_genres'] = load_genres(df)
df = df.dropna(subset=['text']).copy()
df = df[df['text'].astype(str).str.strip().ne('') & df['parsed_genres'].map(bool)].reset_index(drop=True)
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df['parsed_genres']).astype(np.uint8)
print('Rows:', len(df), 'Genres:', len(mlb.classes_))
print('Genres:', list(mlb.classes_))

Rows: 10523 Genres: 20
Genres: ['Action', 'Adventure', 'Avant Garde', 'Boys Love', 'Comedy', 'Drama', 'Ecchi', 'Erotica', 'Fantasy', 'Girls Love', 'Gourmet', 'Hentai', 'Horror', 'Mystery', 'Romance', 'Sci-Fi', 'Slice of Life', 'Sports', 'Supernatural', 'Suspense']


In [3]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text'].astype(str).to_numpy(), Y, test_size=0.2, random_state=SEED
)
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)
n_components = min(100, min(X_train_tfidf.shape) - 1)
if n_components < 1:
    raise ValueError('Insufficient observations/features for TruncatedSVD')
svd = TruncatedSVD(n_components=n_components, random_state=SEED)
X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)
print('Training:', X_train_svd.shape, 'Test:', X_test_svd.shape)

Training: (8418, 100) Test: (2105, 100)


In [4]:
X_aug, y_aug, generated_for, skipped = mlsmote(X_train_svd, y_train, k=K, random_state=SEED)
print('Synthetic observations:', len(y_aug) - len(y_train))
print('MeanIR before/after:', imbalance(y_train)[1], imbalance(y_aug)[1])
print('Skipped singleton minority labels:', list(mlb.classes_[skipped]))

Synthetic observations: 282
MeanIR before/after: 12.682309488056138 8.376707090085764
Skipped singleton minority labels: []


In [5]:
pred_baseline = predict_per_genre(X_train_svd, y_train, X_test_svd, threshold=THRESHOLD)
pred_mlsmote = predict_per_genre(X_aug, y_aug, X_test_svd, threshold=THRESHOLD)
results = pd.DataFrame([
    {'Method': name,
     'Macro F1': f1_score(y_test, pred, average='macro', zero_division=0),
     'Micro F1': f1_score(y_test, pred, average='micro', zero_division=0),
     'Macro Precision': precision_score(y_test, pred, average='macro', zero_division=0),
     'Macro Recall': recall_score(y_test, pred, average='macro', zero_division=0),
     'Avg Predicted Genres': pred.sum(axis=1).mean()}
    for name, pred in [('Baseline', pred_baseline), ('MLSMOTE', pred_mlsmote)]
])
display(results)

,Method,Macro F1,Micro F1,Macro Precision,Macro Recall,Avg Predicted Genres
0,Baseline,0.260124,0.520970,0.397094,0.240924,1.893587
1,MLSMOTE,0.257564,0.519653,0.398497,0.236890,1.857957


In [6]:
genre_report = pd.DataFrame({
    'Genre': mlb.classes_,
    'Training Count': y_train.sum(axis=0),
    'After MLSMOTE': y_aug.sum(axis=0),
    'Directly Seeded': generated_for,
    'Baseline F1': f1_score(y_test, pred_baseline, average=None, zero_division=0),
    'MLSMOTE F1': f1_score(y_test, pred_mlsmote, average=None, zero_division=0),
})
genre_report['Added'] = genre_report['After MLSMOTE'] - genre_report['Training Count']
genre_report['MLSMOTE Improvement'] = genre_report['MLSMOTE F1'] - genre_report['Baseline F1']
display(genre_report.sort_values('Training Count'))

,Genre,Training Count,After MLSMOTE,Directly Seeded,Baseline F1,MLSMOTE F1,Added,MLSMOTE Improvement
7,Erotica,40,85,40,0.000000,0.000000,45,0.000000
9,Girls Love,60,120,60,0.000000,0.000000,60,0.000000
3,Boys Love,89,202,89,0.000000,0.000000,113,0.000000
10,Gourmet,93,186,93,0.000000,0.000000,93,0.000000
19,Suspense,269,269,0,0.000000,0.000000,0,0.000000
17,Sports,285,285,0,0.400000,0.400000,0,0.000000
2,Avant Garde,290,292,0,0.304762,0.271845,2,-0.032917
12,Horror,292,292,0,0.000000,0.000000,0,0.000000
16,Slice of Life,403,431,0,0.000000,0.000000,28,0.000000
11,Hentai,411,411,0,0.303448,0.291667,0,-0.011782
